# Sesión 6: Comparación Final y Dashboard
## ¿Cuál es el Mejor Modelo para Predecir Resultados de Fútbol?

**Objetivo**: Comparar los 4 modelos entrenados, analizar casos difíciles, y dar recomendaciones finales.

**Duración**: ~60 minutos

**Pregunta**: Ahora que tenemos 4 modelos, ¿cuál deberíamos usar y cuándo?

## 1. Configuración y Carga de Datos

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Librerías de machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

# Cargar dataset
data_path = Path('../data/datos_liga_futbol.csv')
df = pd.read_csv(data_path)

print(f"✓ Dataset cargado: {df.shape[0]} partidos")
print(f"✓ Todas las librerías importadas")

## 2. Preprocesamiento y División de Datos

In [ ]:
# Feature engineering (mismo que todas las sesiones anteriores)
df['Diferencia_Habilidad'] = df['Habilidad_Local'] - df['Habilidad_Visitante']
df['Diferencia_Racha'] = df['Racha_Local'] - df['Racha_Visitante']
df['Ratio_Habilidad'] = df['Habilidad_Local'] / df['Habilidad_Visitante']

# Features
features = [
    'Habilidad_Local',
    'Habilidad_Visitante',
    'Racha_Local',
    'Racha_Visitante',
    'Diferencia_Habilidad',
    'Diferencia_Racha',
    'Ratio_Habilidad'
]

X = df[features]
y = df['Resultado']

# Train/test split (MISMO que en todas las sesiones)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print("✓ Features creados")
print(f"  Train set: {len(X_train)} partidos")
print(f"  Test set: {len(X_test)} partidos")

## 3. Entrenamiento de Todos los Modelos

Entrenaremos los 4 modelos con los mismos hiperparámetros que usamos en las sesiones 2-5.

In [ ]:
print("⏳ Entrenando los 4 modelos...\n")

# 1. Regresión Logística
print("  [1/4] Regresión Logística...")
model_logistic = LogisticRegression(max_iter=1000, random_state=42)
model_logistic.fit(X_train, y_train)

# 2. Árbol de Decisión
print("  [2/4] Árbol de Decisión...")
model_tree = DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42)
model_tree.fit(X_train, y_train)

# 3. Random Forest
print("  [3/4] Random Forest...")
model_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)

# 4. XGBoost
print("  [4/4] XGBoost...")
model_xgb = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss',
    use_label_encoder=False
)
model_xgb.fit(X_train, y_train)

print("\n✓ Los 4 modelos han sido entrenados exitosamente")

## 4. Dashboard de Métricas Comparativas

In [ ]:
# Diccionario de modelos
models = {
    'Regresión Logística': model_logistic,
    'Árbol de Decisión': model_tree,
    'Random Forest': model_rf,
    'XGBoost': model_xgb
}

# Calcular métricas para todos los modelos
results = []

for name, model in models.items():
    # Predicciones
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Métricas
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    f1_macro = f1_score(y_test, y_test_pred, average='macro')
    f1_weighted = f1_score(y_test, y_test_pred, average='weighted')
    
    results.append({
        'Modelo': name,
        'Accuracy Train': train_acc,
        'Accuracy Test': test_acc,
        'F1 Macro': f1_macro,
        'F1 Weighted': f1_weighted,
        'Overfitting': train_acc - test_acc
    })

# DataFrame de resultados
df_results = pd.DataFrame(results)

print("="*100)
print("DASHBOARD DE MÉTRICAS - COMPARACIÓN COMPLETA")
print("="*100)
print(df_results.to_string(index=False))

# Dashboard visual
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Accuracy Test (Principal)
ax1 = fig.add_subplot(gs[0, :])
colors = ['#3498db', '#2ecc71', '#9b59b6', '#e74c3c']
bars = ax1.bar(df_results['Modelo'], df_results['Accuracy Test'], color=colors, alpha=0.8, edgecolor='black')
ax1.set_ylabel('Accuracy', fontsize=13, fontweight='bold')
ax1.set_title('🏆 ACCURACY EN TEST SET (Métrica Principal)', fontsize=16, fontweight='bold')
ax1.set_ylim([0, 1])
ax1.grid(axis='y', alpha=0.3)
ax1.tick_params(axis='x', rotation=15)
# Valores sobre barras
for bar, val in zip(bars, df_results['Accuracy Test']):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{val:.3f}\n({val*100:.1f}%)',
             ha='center', va='bottom', fontweight='bold', fontsize=11)

# 2. Train vs Test
ax2 = fig.add_subplot(gs[1, 0])
x = np.arange(len(df_results))
width = 0.35
ax2.bar(x - width/2, df_results['Accuracy Train'], width, label='Train', alpha=0.8, color='lightblue')
ax2.bar(x + width/2, df_results['Accuracy Test'], width, label='Test', alpha=0.8, color='salmon')
ax2.set_ylabel('Accuracy', fontsize=11)
ax2.set_title('Train vs Test', fontsize=13, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(df_results['Modelo'], rotation=20, ha='right', fontsize=9)
ax2.legend()
ax2.set_ylim([0, 1])
ax2.grid(axis='y', alpha=0.3)

# 3. F1 Scores
ax3 = fig.add_subplot(gs[1, 1])
ax3.bar(x - width/2, df_results['F1 Macro'], width, label='F1 Macro', alpha=0.8)
ax3.bar(x + width/2, df_results['F1 Weighted'], width, label='F1 Weighted', alpha=0.8)
ax3.set_ylabel('F1 Score', fontsize=11)
ax3.set_title('F1 Scores', fontsize=13, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(df_results['Modelo'], rotation=20, ha='right', fontsize=9)
ax3.legend()
ax3.set_ylim([0, 1])
ax3.grid(axis='y', alpha=0.3)

# 4. Overfitting
ax4 = fig.add_subplot(gs[1, 2])
colors_over = ['green' if x < 0.05 else 'orange' if x < 0.1 else 'red' 
               for x in df_results['Overfitting']]
ax4.bar(df_results['Modelo'], df_results['Overfitting'], color=colors_over, alpha=0.7)
ax4.set_ylabel('Diferencia', fontsize=11)
ax4.set_title('Overfitting (Train-Test)', fontsize=13, fontweight='bold')
ax4.axhline(0.05, color='green', linestyle='--', alpha=0.4, label='Excelente')
ax4.axhline(0.1, color='orange', linestyle='--', alpha=0.4, label='Aceptable')
ax4.legend(fontsize=8)
ax4.tick_params(axis='x', rotation=20, labelsize=9)
ax4.grid(axis='y', alpha=0.3)

# 5. Ranking
ax5 = fig.add_subplot(gs[2, :])
df_sorted = df_results.sort_values('Accuracy Test', ascending=True)
colors_rank = ['#FFD700' if i == len(df_sorted)-1 else '#C0C0C0' if i == len(df_sorted)-2 
               else '#CD7F32' if i == len(df_sorted)-3 else '#708090' 
               for i in range(len(df_sorted))]
bars_rank = ax5.barh(df_sorted['Modelo'], df_sorted['Accuracy Test'], 
                     color=colors_rank, alpha=0.8, edgecolor='black', linewidth=2)
ax5.set_xlabel('Accuracy Test', fontsize=13, fontweight='bold')
ax5.set_title('🥇 RANKING FINAL DE MODELOS', fontsize=15, fontweight='bold')
ax5.set_xlim([0, 1])
ax5.grid(axis='x', alpha=0.3)
# Medallas
medals = ['🥇', '🥈', '🥉', '4️⃣']
for i, (bar, val) in enumerate(zip(bars_rank, df_sorted['Accuracy Test'])):
    ax5.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.3f} {medals[len(df_sorted)-1-i]}',
             va='center', fontweight='bold', fontsize=12)

plt.suptitle('DASHBOARD COMPARATIVO - 4 MODELOS DE PREDICCIÓN', 
             fontsize=18, fontweight='bold', y=0.995)
plt.show()

# Análisis
print("\n💡 ANÁLISIS DEL DASHBOARD:")
best_idx = df_results['Accuracy Test'].idxmax()
best = df_results.loc[best_idx]
print(f"\n  🏆 CAMPEÓN: {best['Modelo']}")
print(f"     Accuracy Test: {best['Accuracy Test']:.3f} ({best['Accuracy Test']*100:.1f}%)")
print(f"     F1 Score (macro): {best['F1 Macro']:.3f}")
print(f"     Overfitting: {best['Overfitting']:.3f} ({'Bajo' if best['Overfitting'] < 0.1 else 'Moderado'})")

## 5. Matrices de Confusión Lado a Lado

In [ ]:
# Crear matrices de confusión para cada modelo
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('MATRICES DE CONFUSIÓN - COMPARACIÓN', fontsize=16, fontweight='bold')

model_list = [
    ('Regresión Logística', model_logistic, 'Blues'),
    ('Árbol de Decisión', model_tree, 'Greens'),
    ('Random Forest', model_rf, 'Purples'),
    ('XGBoost', model_xgb, 'Oranges')
]

for idx, (name, model, cmap) in enumerate(model_list):
    row = idx // 2
    col = idx % 2
    ax = axes[row, col]
    
    # Predicciones y matriz
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
    accuracy = accuracy_score(y_test, y_pred)
    
    # Heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=model.classes_,
                yticklabels=model.classes_,
                cbar_kws={'label': 'Partidos'})
    ax.set_title(f'{name}\nAccuracy: {accuracy:.3f} ({accuracy*100:.1f}%)', 
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicción', fontsize=11)
    ax.set_ylabel('Real', fontsize=11)

plt.tight_layout()
plt.show()

print("\n📊 INTERPRETACIÓN DE LAS MATRICES:")
print("  - Diagonal principal: ACIERTOS del modelo")
print("  - Fuera de diagonal: ERRORES del modelo")
print("  - Números más altos en diagonal = Mejor modelo")
print("\n  💡 Observa qué clase es más difícil de predecir para cada modelo")

## 6. Análisis por Clase (¿Qué Resultado es Más Difícil de Predecir?)

In [ ]:
# Accuracy por clase para cada modelo
from sklearn.metrics import precision_recall_fscore_support

class_performance = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_test, y_pred, average=None, labels=model.classes_
    )
    
    for i, clase in enumerate(model.classes_):
        class_performance.append({
            'Modelo': name,
            'Clase': clase,
            'Precision': precision[i],
            'Recall': recall[i],
            'F1-Score': f1[i],
            'Support': support[i]
        })

df_class = pd.DataFrame(class_performance)

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

clases = model_logistic.classes_
x = np.arange(len(models))
width = 0.25

metrics = ['Precision', 'Recall', 'F1-Score']
colors_class = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for idx, metric in enumerate(metrics):
    ax = axes[idx]
    
    for i, clase in enumerate(clases):
        data = df_class[(df_class['Clase'] == clase)][metric].values
        ax.bar(x + i*width, data, width, label=clase, alpha=0.8)
    
    ax.set_ylabel(metric, fontsize=12, fontweight='bold')
    ax.set_title(f'{metric} por Clase', fontsize=13, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels([m for m in models.keys()], rotation=15, ha='right', fontsize=10)
    ax.legend(title='Clase', fontsize=9)
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('DESEMPEÑO POR CLASE - ¿Cuál es Más Difícil de Predecir?', 
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Análisis de dificultad por clase
print("\n📊 ANÁLISIS DE DIFICULTAD POR CLASE:")
print("="*70)

avg_by_class = df_class.groupby('Clase')[['Precision', 'Recall', 'F1-Score']].mean()
avg_by_class = avg_by_class.sort_values('F1-Score', ascending=False)

print("\nF1-Score Promedio por Clase (sobre los 4 modelos):")
print(avg_by_class)

print(f"\n💡 CONCLUSIONES:")
easiest_class = avg_by_class.index[0]
hardest_class = avg_by_class.index[-1]
print(f"  - Clase MÁS FÁCIL de predecir: {easiest_class}")
print(f"    F1-Score promedio: {avg_by_class.loc[easiest_class, 'F1-Score']:.3f}")
print(f"\n  - Clase MÁS DIFÍCIL de predecir: {hardest_class}")
print(f"    F1-Score promedio: {avg_by_class.loc[hardest_class, 'F1-Score']:.3f}")
print(f"\n  - Típicamente los EMPATES son más difíciles de predecir")
print(f"    porque son eventos más raros y menos predecibles")

## 7. Casos Difíciles: ¿Dónde Fallan los Modelos?

In [ ]:
# Encontrar casos donde TODOS los modelos fallan
predictions = {}
for name, model in models.items():
    predictions[name] = model.predict(X_test)

# Crear DataFrame con predicciones
df_test = df.loc[X_test.index].copy()
for name in models.keys():
    df_test[f'Pred_{name}'] = predictions[name]

# Casos donde TODOS fallan
df_test['Todos_Correctos'] = True
df_test['Todos_Incorrectos'] = True
df_test['Num_Correctos'] = 0

for name in models.keys():
    pred_col = f'Pred_{name}'
    correct = df_test[pred_col] == df_test['Resultado']
    df_test['Todos_Correctos'] &= correct
    df_test['Todos_Incorrectos'] &= ~correct
    df_test['Num_Correctos'] += correct.astype(int)

# Estadísticas
print("ANÁLISIS DE CASOS DIFÍCILES:")
print("="*80)
print(f"\nTotal de partidos en test set: {len(df_test)}")
print(f"\nPartidos donde TODOS los modelos acertaron: {df_test['Todos_Correctos'].sum()}")
print(f"Partidos donde TODOS los modelos fallaron: {df_test['Todos_Incorrectos'].sum()}")
print(f"\nDistribución de aciertos:")
print(df_test['Num_Correctos'].value_counts().sort_index())

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribución de aciertos
counts = df_test['Num_Correctos'].value_counts().sort_index()
colors_bar = ['red' if x == 0 else 'orange' if x < 3 else 'lightgreen' if x < 4 else 'green' 
              for x in counts.index]
axes[0].bar(counts.index, counts.values, color=colors_bar, alpha=0.8, edgecolor='black')
axes[0].set_xlabel('Número de Modelos que Acertaron', fontsize=12)
axes[0].set_ylabel('Cantidad de Partidos', fontsize=12)
axes[0].set_title('Distribución de Consenso entre Modelos', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(5))
axes[0].grid(axis='y', alpha=0.3)
# Añadir valores
for i, v in zip(counts.index, counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Aciertos por resultado
consensus_by_result = df_test.groupby('Resultado')['Num_Correctos'].mean()
colors_result = ['#FF6B6B', '#4ECDC4', '#45B7D1']
axes[1].bar(consensus_by_result.index, consensus_by_result.values, 
            color=colors_result, alpha=0.8, edgecolor='black')
axes[1].set_xlabel('Tipo de Resultado', fontsize=12)
axes[1].set_ylabel('Promedio de Modelos que Aciertan', fontsize=12)
axes[1].set_title('Dificultad de Predicción por Tipo de Resultado', fontsize=13, fontweight='bold')
axes[1].set_ylim([0, 4])
axes[1].grid(axis='y', alpha=0.3)
# Añadir valores
for i, (label, v) in enumerate(zip(consensus_by_result.index, consensus_by_result.values)):
    axes[1].text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Ejemplos de casos difíciles
print("\n\n❌ EJEMPLOS DE CASOS DONDE TODOS FALLARON:")
print("="*80)

casos_dificiles = df_test[df_test['Todos_Incorrectos']].head(3)

if len(casos_dificiles) > 0:
    for idx, partido in casos_dificiles.iterrows():
        print(f"\n🔍 PARTIDO: {partido['Equipo_Local']} vs {partido['Equipo_Visitante']}")
        print(f"   Habilidades: {partido['Habilidad_Local']} vs {partido['Habilidad_Visitante']}")
        print(f"   Rachas: {partido['Racha_Local']} vs {partido['Racha_Visitante']}")
        print(f"   Resultado REAL: {partido['Resultado']}")
        print(f"   Predicciones:")
        for name in models.keys():
            print(f"      {name:20s}: {partido[f'Pred_{name}']}")
        print("-" * 80)
else:
    print("   ¡Excelente! No hay casos donde todos los modelos fallen.")

print("\n\n✅ EJEMPLOS DE CASOS DONDE TODOS ACERTARON:")
print("="*80)

casos_faciles = df_test[df_test['Todos_Correctos']].head(3)

for idx, partido in casos_faciles.iterrows():
    print(f"\n✓ PARTIDO: {partido['Equipo_Local']} vs {partido['Equipo_Visitante']}")
    print(f"   Habilidades: {partido['Habilidad_Local']} vs {partido['Habilidad_Visitante']}")
    print(f"   Diferencia: {partido['Diferencia_Habilidad']:.1f}")
    print(f"   Resultado: {partido['Resultado']} ← TODOS ACERTARON")
    print("-" * 80)

## 8. Tiempo de Entrenamiento y Predicción

In [ ]:
import time

# Medir tiempos
timing_results = []

for name, ModelClass in [
    ('Regresión Logística', LogisticRegression),
    ('Árbol de Decisión', DecisionTreeClassifier),
    ('Random Forest', RandomForestClassifier),
    ('XGBoost', XGBClassifier)
]:
    # Tiempo de entrenamiento
    if name == 'Regresión Logística':
        model_temp = ModelClass(max_iter=1000, random_state=42)
    elif name == 'Árbol de Decisión':
        model_temp = ModelClass(max_depth=5, random_state=42)
    elif name == 'Random Forest':
        model_temp = ModelClass(n_estimators=100, random_state=42, n_jobs=1)
    else:  # XGBoost
        model_temp = ModelClass(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False)
    
    start = time.time()
    model_temp.fit(X_train, y_train)
    train_time = time.time() - start
    
    # Tiempo de predicción
    start = time.time()
    _ = model_temp.predict(X_test)
    pred_time = time.time() - start
    
    timing_results.append({
        'Modelo': name,
        'Tiempo Entrenamiento (s)': train_time,
        'Tiempo Predicción (s)': pred_time,
        'Total (s)': train_time + pred_time
    })

df_timing = pd.DataFrame(timing_results)

print("⏱️  ANÁLISIS DE TIEMPOS DE EJECUCIÓN:")
print("="*70)
print(df_timing.to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Tiempo de entrenamiento
axes[0].barh(df_timing['Modelo'], df_timing['Tiempo Entrenamiento (s)'], 
             color=['#3498db', '#2ecc71', '#9b59b6', '#e74c3c'], alpha=0.8)
axes[0].set_xlabel('Tiempo (segundos)', fontsize=12)
axes[0].set_title('Tiempo de Entrenamiento', fontsize=13, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Tiempo total
axes[1].barh(df_timing['Modelo'], df_timing['Total (s)'], 
             color=['#3498db', '#2ecc71', '#9b59b6', '#e74c3c'], alpha=0.8)
axes[1].set_xlabel('Tiempo (segundos)', fontsize=12)
axes[1].set_title('Tiempo Total (Entrenamiento + Predicción)', fontsize=13, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('⏱️  COMPARACIÓN DE VELOCIDAD', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 OBSERVACIONES:")
fastest = df_timing.loc[df_timing['Total (s)'].idxmin(), 'Modelo']
slowest = df_timing.loc[df_timing['Total (s)'].idxmax(), 'Modelo']
print(f"  - Modelo MÁS RÁPIDO: {fastest}")
print(f"  - Modelo MÁS LENTO: {slowest}")
print(f"  - Modelos de ensemble (RF, XGBoost) son más lentos pero más precisos")
print(f"  - Para producción: considerar trade-off velocidad vs precisión")

## 9. Recomendaciones Finales

### 🎯 ¿Cuándo Usar Cada Modelo?

In [ ]:
# Crear tabla de recomendaciones
recommendations = pd.DataFrame([
    {
        'Modelo': 'Regresión Logística',
        'Usar Cuando': 'Necesitas velocidad y simplicidad',
        'Ventajas': 'Rápido, fácil de interpretar coeficientes',
        'Desventajas': 'Solo captura relaciones lineales',
        'Caso de Uso': 'Baseline, producción con requisitos de velocidad'
    },
    {
        'Modelo': 'Árbol de Decisión',
        'Usar Cuando': 'Necesitas máxima interpretabilidad',
        'Ventajas': 'Muy interpretable, muestra reglas claras',
        'Desventajas': 'Propenso a overfitting, menos preciso',
        'Caso de Uso': 'Explicar decisiones a stakeholders'
    },
    {
        'Modelo': 'Random Forest',
        'Usar Cuando': 'Buscas balance precisión/velocidad',
        'Ventajas': 'Robusto, reduce overfitting, preciso',
        'Desventajas': 'Menos interpretable, más lento',
        'Caso de Uso': 'Producción general, buen default'
    },
    {
        'Modelo': 'XGBoost',
        'Usar Cuando': 'Necesitas máxima precisión',
        'Ventajas': 'Mejor accuracy, ganador de competencias',
        'Desventajas': 'Más complejo, requiere tuning',
        'Caso de Uso': 'Competencias ML, proyectos críticos'
    }
])

print("="*120)
print("GUÍA DE SELECCIÓN DE MODELOS")
print("="*120)
for _, row in recommendations.iterrows():
    print(f"\n🔹 {row['Modelo'].upper()}")
    print(f"   📌 Usar cuando: {row['Usar Cuando']}")
    print(f"   ✅ Ventajas: {row['Ventajas']}")
    print(f"   ⚠️  Desventajas: {row['Desventajas']}")
    print(f"   💼 Caso de uso: {row['Caso de Uso']}")
    print("-" * 120)

print("\n\n🎓 RESUMEN EJECUTIVO:")
print("="*80)

best_model = df_results.loc[df_results['Accuracy Test'].idxmax(), 'Modelo']
best_acc = df_results['Accuracy Test'].max()

print(f"\n📊 RESULTADOS DEL PROYECTO:")
print(f"   - Dataset: {len(df)} partidos de fútbol")
print(f"   - Features: {len(features)} variables predictoras")
print(f"   - Modelos evaluados: 4 (Logística, Árbol, Random Forest, XGBoost)")
print(f"   - Mejor modelo: {best_model} ({best_acc*100:.1f}% accuracy)")

print(f"\n🎯 PARA ESTE DATASET:")
if best_model == 'XGBoost' or best_model == 'Random Forest':
    print(f"   ✓ Los modelos de ensemble (RF/XGBoost) superaron a los simples")
    print(f"   ✓ La complejidad adicional valió la pena en precisión")
    print(f"   → RECOMENDACIÓN: Usar {best_model} en producción")
else:
    print(f"   ⚠️  Un modelo simple ({best_model}) fue suficiente")
    print(f"   ✓ En datasets pequeños, la simplicidad puede ganar")
    print(f"   → RECOMENDACIÓN: {best_model} es suficiente para este caso")

print(f"\n💡 LECCIONES APRENDIDAS:")
print(f"   1. Siempre empieza con un modelo simple (baseline)")
print(f"   2. Compara múltiples modelos con los MISMOS datos (random_state=42)")
print(f"   3. No hay un modelo "mejor" universal - depende del problema")
print(f"   4. Considera el trade-off: precisión vs velocidad vs interpretabilidad")
print(f"   5. Feature engineering a menudo importa MÁS que el modelo elegido")

print("\n" + "="*80)

## 10. Resumen Final del Curso

### 🎓 Lo que Aprendimos en 6 Sesiones

#### **Sesión 1: EDA**
- Exploración de datos de fútbol
- Identificación de variables clave
- Calidad de datos y distribuciones

#### **Sesión 2: Regresión Logística**
- Modelo baseline lineal
- Feature engineering (3 nuevas variables)
- Interpretación de coeficientes

#### **Sesión 3: Árboles de Decisión**
- Reglas tipo "si-entonces"
- Visualización de decisiones
- Alta interpretabilidad

#### **Sesión 4: Random Forest**
- Ensemble learning (Bagging)
- 100 árboles votando juntos
- Reducción de overfitting

#### **Sesión 5: XGBoost**
- Gradient Boosting (aprender de errores)
- Algoritmo ganador de competencias
- Máxima precisión

#### **Sesión 6: Comparación Final** ← Estamos aquí
- Dashboard comparativo completo
- Análisis de casos difíciles
- Recomendaciones para producción

---

### 🚀 Próximos Pasos

1. **Experimentar con hiperparámetros**: Ajustar los modelos para mejorar accuracy
2. **Crear más features**: Probar otras variables derivadas
3. **Cross-validation**: Validar resultados con k-fold
4. **Nuevos modelos**: Probar SVM, Neural Networks, etc.
5. **Deployar en producción**: Crear API para hacer predicciones en tiempo real

---

### 🎉 ¡FELICITACIONES!

Has completado un curso completo de Machine Learning aplicado:
- ✅ 6 sesiones hands-on
- ✅ 4 modelos diferentes
- ✅ Pipeline completo: EDA → Modelado → Evaluación → Comparación
- ✅ Skills aplicables a cualquier problema de clasificación

**¡Ahora tienes las herramientas para resolver problemas reales con ML!** 🎓🚀

In [ ]:
# Certificado de finalización
print("="*80)
print(" "*20 + "🎓 CERTIFICADO DE FINALIZACIÓN 🎓")
print("="*80)
print()
print("  Felicitaciones por completar:")
print()
print("  📚 CURSO: Modelado Predictivo con Machine Learning")
print("  🎯 PROYECTO: Predicción de Resultados de Partidos de Fútbol")
print()
print("  ✅ Módulos Completados:")
print("     1. Análisis Exploratorio de Datos (EDA)")
print("     2. Regresión Logística")
print("     3. Árboles de Decisión")
print("     4. Random Forest")
print("     5. XGBoost")
print("     6. Comparación y Dashboard Final")
print()
print("  🏆 Habilidades Adquiridas:")
print("     • Feature Engineering")
print("     • Entrenamiento de modelos de ML")
print("     • Evaluación y métricas")
print("     • Comparación de modelos")
print("     • Interpretación de resultados")
print("     • Visualización de datos")
print()
print(f"  📊 Mejor Modelo Logrado: {best_model}")
print(f"  🎯 Accuracy Final: {best_acc:.3f} ({best_acc*100:.1f}%)")
print()
print("  🚀 ¡Listo para aplicar Machine Learning a problemas reales!")
print()
print("="*80)
print(" "*25 + "¡EXCELENTE TRABAJO! 🎉")
print("="*80)